<a href="https://colab.research.google.com/github/sbooeshaghi/llmarkers/blob/main/analysis/token_aln.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# lets see if I can do the same with a tokenization scheme

In [1]:
!pip install --quiet tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 13.8 MB/s eta 0:00:00


# Token based

In [1]:
import tiktoken

In [8]:
target = " sina"
source = "hello my name is sina. his name is frank".replace("\n", " ")

In [35]:
import re

def tokenize_whitespace_with_offsets(text):
    """Tokenizes text on whitespace and returns tokens with their character start and end positions."""
    tokens = []

    for word in re.finditer(r'\S+', text):  # word non-whitespace sequences
        token = word.group()
        start_idx = word.start()
        end_idx = word.end()

        tokens.append({
            "token": token,
            "start_idx": start_idx,
            "end_idx": end_idx
        })

    return tokens

def tokenize_with_offsets(text, encoding="cl100k_base"):
  """Tokenizes text and returns tokens with their character start positions."""
  enc = tiktoken.get_encoding(encoding)
  tokens = []

  etks = enc.encode(text)
  dtks, ptks = enc.decode_with_offsets(etks)

  assert len(etks) == len(ptks)

  for t,p in zip(etks, ptks):
    token = enc.decode_single_token_bytes(t).decode("utf-8")
    tobj = {
      "token": token,
      "enc_token": t,
      "start_idx": p,
      "end_idx": p + len(token)
    }
    tokens.append(tobj)
  return tokens

from collections import defaultdict
def index_source(source):
  source_tokens = tokenize_with_offsets(source)
  source_words = tokenize_whitespace_with_offsets(source)

  # tokens = map_tokens_to_words(source_tokens, source_words)
  tokens = source_tokens

  source_index = defaultdict(list)
  for i, token in enumerate(tokens):
    source_index[token["enc_token"]].append(token)
  return source_index

def align_target(target, idx):
  pos = defaultdict(list)
  tokens = tokenize_with_offsets(target)
  aln = []
  for token_obj in tokens:
    enc_token = token_obj["enc_token"]
    if enc_token in idx.keys():
      aln.append({enc_token: idx[enc_token]})
  return aln

def deduplicate_alignments(aln):
  pos = []
  start_idx = 0

  for d in aln:
      k, v = next(iter(d.items()))
      sorted_targets = sorted(v, key=lambda x: x["start_idx"])

      # Use next with a generator expression to find the first valid match
      valid_target = next((t for t in sorted_targets if t["start_idx"] >= start_idx), None)

      if valid_target:
          start_idx = valid_target["start_idx"]
          pos.append(valid_target)

  return pos

def group_contiguous_tokens(pos):
  group = []
  g = []
  for tk in pos:
    if len(g) == 0:
      g.append(tk)
    elif g[-1]["end_idx"] == tk["start_idx"]:
      g.append(tk)
    elif g[-1]["end_idx"] < tk["start_idx"]:
      group.append({reconstruct_target_by_token("", g): g})
      g = [tk]
  group.append({reconstruct_target_by_token("", g): g})
  return group

def reconstruct_target_by_token(source, pos):
  # reconstruct the target by joining the tokens
  return "".join([i["token"] for i in pos])

def reconstruct_target_by_idx(source, pos):
  # reconstruct the target by joining the tokens
  return "".join([source[i["start_idx"]:i["end_idx"]] for i in pos])

def reconstruct_target_by_strip_token(source, pos):
  # reconstruct the target by joining the tokens
  return " ".join([i["strip_token"] for i in pos])

def align(source, target):
  idx = index_source(source)
  aln = align_target(target, idx)
  pos = deduplicate_alignments(aln)
  grp = group_contiguous_tokens(pos)
  found = reconstruct_target_by_idx(source, pos)
  return (found, grp)

In [44]:
source = """For example, mASPC2 and hASPC2 are characterized by high expression of Aldh1a3 and ALDH1A3,
respectively, and strongly resemble previously identified early multipotent progenitor cells that
reside in the reticular interstitium of the fat pad5.""".replace("\n", " ")

target = " and strongly resemble previously identified early multipotent progenitor"

idx = index_source(source)
aln = align_target(target, idx)
pos = deduplicate_alignments(aln)
grp = group_contiguous_tokens(pos)
found = reconstruct_target_by_idx(source, pos)
print(f"target: {target}")
print(f"found : {found}")

target:  and strongly resemble previously identified early multipotent progenitor
found :  and strongly resemble previously identified early multipotent progenitor


In [45]:
target =  " male populations"
source = """After doublet removal and quality filtering, we considered a total of 197,721 cells (106,469 from PG and 91,252 from ING),
identifying all cell types observed in human WAT (Fig. 2c, d, Supplementary Table 2) with the addition of
distinct male and female epithelial populations (Dcdc2a+ and Erbb4+, respectively)""".replace("\n", " ")

found, grp = align(source, target)
print(f"target: {target}")
print(f"found : {found}")
print(grp)

target:  male populations
found :  male populations
[{' male': [{'token': ' male', 'enc_token': 8762, 'start_idx': 239, 'end_idx': 244}]}, {' populations': [{'token': ' populations', 'enc_token': 22673, 'start_idx': 266, 'end_idx': 278}]}]


In [46]:
target = " hASPC2"
source = """For example, mASPC2 and hASPC2 are characterized by high expression of Aldh1a3 and ALDH1A3,
respectively, and strongly resemble previously identified early multipotent progenitor cells that reside
in the reticular interstitium of the fat pad5."""

found, grp = align(source, target)
print(f"target: {target}")
print(f"found : {found}")
print(grp)

target:  hASPC2
found :  hASPC2
[{' hASPC2': [{'token': ' h', 'enc_token': 305, 'start_idx': 23, 'end_idx': 25}, {'token': 'AS', 'enc_token': 1950, 'start_idx': 25, 'end_idx': 27}, {'token': 'PC', 'enc_token': 4977, 'start_idx': 27, 'end_idx': 29}, {'token': '2', 'enc_token': 17, 'start_idx': 29, 'end_idx': 30}]}]
